# 01 — HDFS + Spark smoke test
Menguji 5 hal: (1) put/ls/fsck, (2) NameNode UI, (3) WebHDFS, (4) PySpark baca/tulis HDFS, (5) fsspec WebHDFS.

Syarat: HDFS sudah jalan (`jps` menampilkan NameNode & DataNode).

In [1]:
import os, subprocess
HADOOP = os.path.expanduser('~/opt/hadoop/bin')
os.environ['JAVA_HOME'] = subprocess.check_output(['/usr/libexec/java_home', '-v', '11']).decode().strip()
os.environ['PATH'] = HADOOP + ':' + os.environ['PATH']
def sh(cmd):
    print('$', cmd); r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr); assert r.returncode == 0, 'command failed'
    return r.stdout
CSV = os.path.abspath('../outputs/tables/smoke.csv')
with open(CSV, 'w') as f:
    f.write('id,value\n' + ''.join(f'{i},{i*i}\n' for i in range(1000)))

## 1. put / ls / fsck

In [2]:
sh('hdfs dfs -mkdir -p /tmp_smoke')
sh(f'hdfs dfs -put -f {CSV} /tmp_smoke/')
sh('hdfs dfs -ls /tmp_smoke')
sh('hdfs fsck /tmp_smoke -files -blocks');

$ hdfs dfs -mkdir -p /tmp_smoke


2026-09-23 05:19:21,739 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable

$ hdfs dfs -put -f /Users/axelnakata/Documents/BigData-Earthquake/outputs/tables/smoke.csv /tmp_smoke/


2026-09-23 05:19:22,387 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable

$ hdfs dfs -ls /tmp_smoke


Found 1 items
-rw-r--r--   1 axelnakata supergroup      10436 2026-09-23 05:19 /tmp_smoke/smoke.csv
2026-09-23 05:19:23,602 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable

$ hdfs fsck /tmp_smoke -files -blocks


FSCK started by axelnakata (auth:SIMPLE) from /127.0.0.1 for path /tmp_smoke at Wed Sep 23 05:19:24 WIB 2026

/tmp_smoke <dir>
/tmp_smoke/smoke.csv 10436 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-96604845-192.168.100.2-1790115507935:blk_1073741825_1001 len=10436 Live_repl=1


Status: HEALTHY
 Number of data-nodes:	1
 Number of racks:		1
 Total dirs:			1
 Total symlinks:		0

Replicated Blocks:
 Total size:	10436 B
 Total files:	1
 Total blocks (validated):	1 (avg. block size 10436 B)
 Minimally replicated blocks:	1 (100.0 %)
 Over-replicated blocks:	0 (0.0 %)
 Under-replicated blocks:	0 (0.0 %)
 Mis-replicated blocks:		0 (0.0 %)
 Default replication factor:	1
 Average block replication:	1.0
 Missing blocks:		0
 Corrupt blocks:		0
 Missing replicas:		0 (0.0 %)
 Blocks queued for replication:	0

Erasure Coded Block Groups:
 Total size:	0 B
 Total files:	0
 Total block groups (validated):	0
 Minimally erasure-coded block groups:	0
 Over-erasure-coded block groups:	0
 Under-er

## 2. NameNode UI
Buka http://localhost:9870 → screenshot halaman *Overview* dan *Utilities → Browse the file system* ke `outputs/screenshots/`.

In [3]:
import urllib.request
print(urllib.request.urlopen('http://localhost:9870').status)

200


## 3. WebHDFS (curl, ikuti redirect ke DataNode)

In [4]:
out = sh('curl -s -L "http://localhost:9870/webhdfs/v1/tmp_smoke/smoke.csv?op=OPEN&user.name=$(whoami)" | head -3')
assert out.startswith('id,value')

$ curl -s -L "http://localhost:9870/webhdfs/v1/tmp_smoke/smoke.csv?op=OPEN&user.name=$(whoami)" | head -3


id,value
0,0
1,1



## 4. PySpark: CSV (HDFS) → Parquet (HDFS) → baca lagi

In [5]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder.master('local[*]').appName('smoke')
         .config('spark.local.dir', os.path.expanduser('~/pnwml-data/tmp/spark-local'))
         .config('spark.hadoop.fs.defaultFS', 'hdfs://localhost:9000')
         .getOrCreate())
df = spark.read.csv('hdfs://localhost:9000/tmp_smoke/smoke.csv', header=True, inferSchema=True)
df.write.mode('overwrite').parquet('hdfs://localhost:9000/tmp_smoke/smoke.parquet')
back = spark.read.parquet('hdfs://localhost:9000/tmp_smoke/smoke.parquet')
print(df.count(), back.count()); assert df.count() == back.count() == 1000
back.show(3)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/23 05:19:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/23 05:19:26 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


1000 1000
+---+-----+
| id|value|
+---+-----+
|  0|    0|
|  1|    1|
|  2|    4|
+---+-----+
only showing top 3 rows



## 5. fsspec WebHDFS (cara Python membaca HDF5 nanti)

In [6]:
import fsspec, getpass
fs = fsspec.filesystem('webhdfs', host='localhost', port=9870, user=getpass.getuser())
with fs.open('/tmp_smoke/smoke.csv', 'rb') as f:
    head = f.read(20)
print(head); assert head.startswith(b'id,value')

b'id,value\n0,0\n1,1\n2,4'


## Bersihkan

In [7]:
spark.stop()
sh('hdfs dfs -rm -r -skipTrash /tmp_smoke'); os.remove(CSV)
print('SMOKE TEST LULUS')

$ hdfs dfs -rm -r -skipTrash /tmp_smoke


Deleted /tmp_smoke
2026-09-23 05:19:31,725 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable

SMOKE TEST LULUS
